# EVAL-001 — frozen EXP-011 on labelled jaNemVect1.1 test

Финальный test-аудит после фиксации logistic champion. Test ground truth запрещено использовать для изменения признаков, C или выбора модели.

In [1]:
# Выполнить первой после restart kernel.
import os
os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")

'2'

In [2]:
from pathlib import Path
import json
import sys
from IPython.display import display
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (CURRENT_DIR, *CURRENT_DIR.parents)
                    if (p / "README.md").is_file() and (p / "src/ml_project").is_dir())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_data import ensure_prepared_split
from ml_project.epic_test_evaluation import (
    EXPECTED_TEST_POSITIONS, load_labeled_test,
)

## 1. Доказать, что модель заморожена

In [3]:
CHAMPION_DIR = PROJECT_ROOT / "artifacts/experiments/EXP-011/run_001"
champion_metadata = json.loads((CHAMPION_DIR / "metadata.json").read_text(encoding="utf-8"))
assert champion_metadata["run"]["decision"] == "adopt"
assert champion_metadata["experiment"]["experiment_id"] == "EXP-011"
assert champion_metadata["parameters"]["selected_variant"] == "EXP-009+GG+TA"
assert float(champion_metadata["parameters"]["selected_C"]) == 0.0001
display(pd.Series({
    "experiment": "EXP-011/run_001",
    "decision": champion_metadata["run"]["decision"],
    "variant": champion_metadata["parameters"]["selected_variant"],
    "C": champion_metadata["parameters"]["selected_C"],
    "notebook_sha256": champion_metadata["run"]["notebook_sha256"],
}, name="frozen_value").to_frame())

,frozen_value
experiment,EXP-011/run_001
decision,adopt
variant,EXP-009+GG+TA
C,0.0001
notebook_sha256,bd0f653189edd23d3f29f281b08bee8e18150cfaca5047...


## 2. Test whitelist и template.test

In [4]:
split = ensure_prepared_split(PROJECT_ROOT)
test_intervals = split.intervals("test")
test_coordinate_summary = pd.Series({
    "contigs": test_intervals["contig"].nunique(),
    "template_interval_rows": len(test_intervals),
    "position_strand_count": int(test_intervals["length_bp"].sum()),
    "first_template_index": int(test_intervals["template_start"].min()),
    "last_template_stop": int(test_intervals["template_stop"].max()),
}, name="value")
display(test_coordinate_summary.to_frame())
display(test_intervals.groupby(["contig", "strand"], observed=True)["length_bp"].sum().rename("positions").reset_index())
assert int(test_coordinate_summary["position_strand_count"]) == EXPECTED_TEST_POSITIONS

,value
contigs,3
template_interval_rows,54846
position_strand_count,72613390
first_template_index,0
last_template_stop,72613390


,contig,strand,positions
0,NC_064037.1,+,13880887
1,NC_064037.1,-,13880887
2,NC_064044.1,+,10989536
3,NC_064044.1,-,10989536
4,NC_064046.1,+,11436272
5,NC_064046.1,-,11436272


## 3. Открыть замороженный test ground truth

Эта операция допустима только потому, что EXP-011 уже сохранён как immutable run.

In [5]:
OPEN_TEST_GROUND_TRUTH = True
if OPEN_TEST_GROUND_TRUTH:
    labeled_test = load_labeled_test(PROJECT_ROOT, split)
    display(labeled_test.diagnostics)
    display(
        labeled_test.positive_targets_table
        .groupby(["contig", "strand"], observed=True)
        .agg(positive_positions=("target", "sum"), total_count=("count", "sum"))
        .reset_index()
    )
else:
    print("Test ground truth остаётся закрытым")

,template_interval_rows,whitelist_interval_rows,test_contigs,plus_positions,minus_positions,test_positions,positive_positions,negative_positions,prevalence,total_csrna_count,ground_truth_file,ground_truth_sha256
0,54846,27423,3,36306695,36306695,72613390,53813,72559577,0.000741,885179,data/raw/unpacked/jaNemVect1.1/ground_truth/cs...,8b1a1d38ef89794ffb710c37d8408efeb09b57341dff47...


,contig,strand,positive_positions,total_count
0,NC_064037.1,+,11053,335162
1,NC_064037.1,-,11851,94968
2,NC_064044.1,+,7382,123936
3,NC_064044.1,-,6430,56233
4,NC_064046.1,+,9231,161608
5,NC_064046.1,-,7866,113272


In [6]:
assert int(labeled_test.diagnostics.iloc[0]["test_positions"]) == 72_613_390
assert int(labeled_test.diagnostics.iloc[0]["positive_positions"]) == 53_813
assert labeled_test.ground_truth_sha256 == "8b1a1d38ef89794ffb710c37d8408efeb09b57341dff475051fcfe4db8a7f76c"
assert set(labeled_test.positive_targets_table["contig"]) == set(test_intervals["contig"])
print("PASS: whitelist.test, template.test и csRNA-test согласованы")

PASS: whitelist.test, template.test и csRNA-test согласованы


# STOP 1

На этом этапе test только загружен и проверен. Признаки EXP-011, predictions и метрики ещё не вычислялись.

## 4. Восстановить frozen EXP-011 rank scorer

Intercept в историческом артефакте не сохранён. Для AP и Spearman он не нужен: единый additive shift не меняет порядок score. Вероятности и log-loss здесь запрещены.

In [7]:
import importlib
import ml_project.epic_test_evaluation as test_eval
import ml_project.epic_experiments.exp_011_gg_ta_complementarity as exp011
importlib.reload(exp011)
importlib.reload(test_eval)
from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_test_evaluation import load_frozen_exp011_rank_scorer
from ml_project.epic_experiments.exp_011_gg_ta_complementarity import (
    count_candidate, count_diagnostics, evaluate_candidate,
)

frozen_scorer = load_frozen_exp011_rank_scorer(CHAMPION_DIR)
display(pd.Series({
    "variant": frozen_scorer.variant,
    "features": len(frozen_scorer.coefficients),
    "coefficient_sha256": frozen_scorer.coefficient_sha256,
    "rank_metrics_only": frozen_scorer.rank_metrics_only,
    "intercept_used_for_ranking": frozen_scorer.intercept,
}, name="frozen_value").to_frame())
assert frozen_scorer.rank_metrics_only and len(frozen_scorer.coefficients) == 4400

,frozen_value
variant,EXP-009+GG+TA
features,4400
coefficient_sha256,020ab68d6ad31de162fdb18b8e144da724cd2f0a677b85...
rank_metrics_only,True
intercept_used_for_ranking,0.0


In [8]:
sequences, fasta_alphabet = load_baseline_sequences(split)
display(fasta_alphabet)
print("Loaded contigs:", len(sequences))

{'A': 54173800,
 'C': 39020211,
 'G': 38944862,
 'N': 34600,
 'T': 54153761,
 'a': 25666946,
 'c': 15841270,
 'g': 15877204,
 'n': 600,
 't': 25705184}

Loaded contigs: 47


## 5. Посчитать exact EXP-011 признаки на test

Модель не обучается. Для каждой test-позиции вычисляется тот же `k6 × GC × CpG × GG × TA` contract, после чего одинаковые строки сворачиваются по test-contig.

In [9]:
RUN_TEST_COUNT = True
if RUN_TEST_COUNT:
    test_counts = count_candidate(
        labeled_test, sequences, "test", verbose=True,
    )
    test_count_diagnostics = count_diagnostics(test_counts)
    display(test_count_diagnostics)
else:
    print("Test feature count закрыт: RUN_TEST_COUNT=False")

[01/06] group=NC_064037.1 NC_064037.1 strand=+ positions=13,880,887 part_rows=2,788,441 pending_group_rows=2,788,441 elapsed=12.3s
[02/06] group=NC_064037.1 NC_064037.1 strand=- positions=13,880,887 part_rows=2,781,356 pending_group_rows=3,502,608 elapsed=24.6s
[03/06] group=NC_064044.1 NC_064044.1 strand=+ positions=10,989,536 part_rows=2,526,734 pending_group_rows=2,526,734 elapsed=35.0s
[04/06] group=NC_064044.1 NC_064044.1 strand=- positions=10,989,536 part_rows=2,535,646 pending_group_rows=3,242,405 elapsed=45.5s
[05/06] group=NC_064046.1 NC_064046.1 strand=+ positions=11,436,272 part_rows=2,587,819 pending_group_rows=2,587,819 elapsed=55.8s
[06/06] group=NC_064046.1 NC_064046.1 strand=- positions=11,436,272 part_rows=2,588,808 pending_group_rows=3,295,401 elapsed=66.4s


,split,group_kind,group,positions,joint_rows,unique_rows_per_position,totals_memory_mib,positive_rows
0,test,contig,NC_064037.1,27761774,3502608,0.126167,53.445683,22904
1,test,contig,NC_064044.1,21979072,3242405,0.147522,49.475300,13812
2,test,contig,NC_064046.1,22872544,3295401,0.144077,50.283955,17097
3,test,all,TOTAL,72613390,10040414,0.138272,153.204937,53813


# STOP 2

После exact test count сначала проверяем позиции, positives и память. AP и Spearman открываются только следующим шагом.

In [10]:
RUN_TEST_EVALUATION = True
if RUN_TEST_EVALUATION:
    test_evaluation = evaluate_candidate(test_counts, frozen_scorer)
    display(test_evaluation.metric_summary)
    display(test_evaluation.per_contig_metrics)
else:
    print("Test metrics закрыты: RUN_TEST_EVALUATION=False")

,segment,variant,average_precision,epic_spearman,positions,positives,prevalence
0,overall,EXP-009+GG+TA,0.002844,0.096185,72613390,53813,0.000741


,contig,variant,average_precision,epic_spearman,positions,positives,prevalence
0,NC_064037.1,EXP-009+GG+TA,0.003087,0.098276,27761774,22904,0.000825
1,NC_064044.1,EXP-009+GG+TA,0.002408,0.107743,21979072,13812,0.000628
2,NC_064046.1,EXP-009+GG+TA,0.002972,0.129850,22872544,17097,0.000747


## 6. Официальные scoring half1 / half2

EPIC Solution публикует AP и Spearman отдельно на `half1` и `half2`, затем усредняет. Full-test метрика полезна для sanity check, но напрямую с их таблицей не сравнивается.

In [11]:
importlib.reload(test_eval)
from ml_project.epic_test_evaluation import load_official_test_halves
official_halves = load_official_test_halves(PROJECT_ROOT, labeled_test)
half_contract = pd.concat(
    [item.diagnostics for item in official_halves.values()], ignore_index=True
)
display(half_contract)
assert int(half_contract["positions"].sum()) == 72_613_390
assert int(half_contract["positive_positions"].sum()) == 53_813

# Сохраняем компактные full-test результаты и освобождаем большие таблицы
# перед двумя дополнительными official-half count.
full_test_metric_summary = test_evaluation.metric_summary.copy()
full_test_per_contig_metrics = test_evaluation.per_contig_metrics.copy()
full_test_positive_predictions = test_evaluation.positive_predictions.copy()
del test_counts, test_evaluation
import gc
gc.collect()

,scoring_half,interval_rows_both_strands,test_contigs,positions,positive_positions,negative_positions,prevalence,total_csrna_count,ground_truth_sha256
0,half1,27498,3,36305148,28964,36276184,0.000798,534373,8b1a1d38ef89794ffb710c37d8408efeb09b57341dff47...
1,half2,27350,3,36308242,24849,36283393,0.000684,350806,8b1a1d38ef89794ffb710c37d8408efeb09b57341dff47...


34

In [12]:
RUN_OFFICIAL_HALF_COUNTS = True
if RUN_OFFICIAL_HALF_COUNTS:
    official_half_counts = {}
    half_count_diagnostics = []
    for half_name, half_data in official_halves.items():
        print(f"Counting {half_name} ...")
        counted = count_candidate(half_data, sequences, "test", verbose=True)
        official_half_counts[half_name] = counted
        local = count_diagnostics(counted)
        local.insert(0, "scoring_half", half_name)
        half_count_diagnostics.append(local)
    half_count_diagnostics = pd.concat(half_count_diagnostics, ignore_index=True)
    display(half_count_diagnostics)
else:
    print("Official half counts закрыты: RUN_OFFICIAL_HALF_COUNTS=False")

Counting half1 ...
[01/06] group=NC_064037.1 NC_064037.1 strand=+ positions=6,939,286 part_rows=2,104,768 pending_group_rows=2,104,768 elapsed=6.0s
[02/06] group=NC_064037.1 NC_064037.1 strand=- positions=6,939,286 part_rows=2,095,799 pending_group_rows=2,769,468 elapsed=12.2s
[03/06] group=NC_064044.1 NC_064044.1 strand=+ positions=5,499,387 part_rows=1,842,498 pending_group_rows=1,842,498 elapsed=17.6s
[04/06] group=NC_064044.1 NC_064044.1 strand=- positions=5,499,387 part_rows=1,848,447 pending_group_rows=2,500,267 elapsed=23.1s
[05/06] group=NC_064046.1 NC_064046.1 strand=+ positions=5,713,901 part_rows=1,952,640 pending_group_rows=1,952,640 elapsed=28.2s
[06/06] group=NC_064046.1 NC_064046.1 strand=- positions=5,713,901 part_rows=1,954,151 pending_group_rows=2,618,740 elapsed=33.5s
Counting half2 ...
[01/06] group=NC_064037.1 NC_064037.1 strand=+ positions=6,941,601 part_rows=2,111,082 pending_group_rows=2,111,082 elapsed=6.3s
[02/06] group=NC_064037.1 NC_064037.1 strand=- positio

,scoring_half,split,group_kind,group,positions,joint_rows,unique_rows_per_position,totals_memory_mib,positive_rows
0,half1,test,contig,NC_064037.1,13878572,2769468,0.199550,42.258854,12692
1,half1,test,contig,NC_064044.1,10998774,2500267,0.227322,38.151173,6900
2,half1,test,contig,NC_064046.1,11427802,2618740,0.229155,39.958927,9372
3,half1,test,all,TOTAL,36305148,7888475,0.217283,120.368954,28964
4,half2,test,contig,NC_064037.1,13883202,2783838,0.200518,42.478123,10212
5,half2,test,contig,NC_064044.1,10980298,2552043,0.232420,38.941212,6912
6,half2,test,contig,NC_064046.1,11444742,2542171,0.222126,38.790577,7725
7,half2,test,all,TOTAL,36308242,7878052,0.216977,120.209911,24849


# STOP 3

Перед официальными half-метриками проверяем, что суммы positions и positives снова равны полному test.

In [ ]:
RUN_OFFICIAL_HALF_EVALUATION = False
if RUN_OFFICIAL_HALF_EVALUATION:
    official_half_evaluations = {
        name: evaluate_candidate(counts, frozen_scorer, include_curve=False)
        for name, counts in official_half_counts.items()
    }
    exp011_official_metrics = pd.concat([
        evaluation.metric_summary.assign(scoring_half=name)
        for name, evaluation in official_half_evaluations.items()
    ], ignore_index=True)
    mean_row = pd.DataFrame([{
        "segment": "official mean of halves",
        "variant": frozen_scorer.variant,
        "average_precision": exp011_official_metrics["average_precision"].mean(),
        "epic_spearman": exp011_official_metrics["epic_spearman"].mean(),
        "positions": exp011_official_metrics["positions"].sum(),
        "positives": exp011_official_metrics["positives"].sum(),
        "prevalence": exp011_official_metrics["positives"].sum() / exp011_official_metrics["positions"].sum(),
        "scoring_half": "mean",
    }])
    exp011_official_metrics = pd.concat([exp011_official_metrics, mean_row], ignore_index=True)
    display(exp011_official_metrics)
else:
    print("Official half metrics закрыты: RUN_OFFICIAL_HALF_EVALUATION=False")